# Swiss Rail Reliability: explore the results
Run the cells from top to bottom (**Shift + Enter**). Tables appear as clean HTML tables, ready for screenshots.

In [2]:
import sys
sys.path.append("/app/src")
from spark_session import PARQUET_DIR, get_spark

spark = get_spark("explore")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/15 20:04:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from spark_session import PARQUET_DIR, OUTPUT_DIR, get_spark

spark = get_spark("notebook")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

def show(df, n=10):
    """Display the first n rows of a Spark table as a nice HTML table."""
    return df.limit(n).toPandas()

def table(name):
    return spark.read.parquet(str(PARQUET_DIR / name))

print("Spark ready")

Spark ready


26/10/15 20:05:11 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


## 1. After ingestion: typed table (explicit schema)

In [4]:
stops = table("stops_typed")
stops.printSchema()
print(f"rows: {stops.count():,}")

root
 |-- trip_id: string (nullable = true)
 |-- operator_id: string (nullable = true)
 |-- operator_abbr: string (nullable = true)
 |-- operator_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- line_id: string (nullable = true)
 |-- line_text: string (nullable = true)
 |-- circulation_id: string (nullable = true)
 |-- vehicle_type: string (nullable = true)
 |-- is_extra_trip: boolean (nullable = true)
 |-- is_cancelled: boolean (nullable = true)
 |-- station_id: integer (nullable = true)
 |-- station_name: string (nullable = true)
 |-- planned_arrival: timestamp (nullable = true)
 |-- actual_arrival: timestamp (nullable = true)
 |-- arrival_status: string (nullable = true)
 |-- planned_departure: timestamp (nullable = true)
 |-- actual_departure: timestamp (nullable = true)
 |-- departure_status: string (nullable = true)
 |-- is_pass_through: boolean (nullable = true)
 |-- sloid: string (nullable = true)
 |-- operating_day: date (nullable = true)



[Stage 1:======================================================>  (22 + 1) / 23]

rows: 36,412,778


### Example: trains at Bern on 1 October (planned vs actual time, status)

In [5]:
show(stops.filter("product = 'Zug' AND station_id = 8507000 AND operating_day = '2026-10-01'")
          .select("operating_day", "line_text", "station_name", "planned_arrival", "actual_arrival", "arrival_status")
          .orderBy("planned_arrival"))

,operating_day,line_text,station_name,planned_arrival,actual_arrival,arrival_status
0,2026-10-01,IR35,Bern,NaT,NaT,None
1,2026-10-01,IR16,Bern,NaT,NaT,None
2,2026-10-01,RE2,Bern,NaT,NaT,None
3,2026-10-01,IR16,Bern,NaT,NaT,None
4,2026-10-01,RE5,Bern,NaT,NaT,None
5,2026-10-01,IR16,Bern,NaT,NaT,None
6,2026-10-01,RE2,Bern,NaT,NaT,None
7,2026-10-01,IC,Bern,NaT,NaT,None
8,2026-10-01,IR35,Bern,NaT,NaT,None
9,2026-10-01,IR16,Bern,NaT,NaT,None


### Ingestion report (rows read, values that could not be converted)

In [5]:
pd.read_csv(OUTPUT_DIR / "ingestion_report.csv")

,table,check,column,count
0,istdaten,rows read,NaN,36412778
1,istdaten,not convertible to date,BETRIEBSTAG -> operating_day,0
2,istdaten,not convertible to boolean,ZUSATZFAHRT_TF -> is_extra_trip,0
3,istdaten,not convertible to boolean,FAELLT_AUS_TF -> is_cancelled,0
4,istdaten,not convertible to int,BPUIC -> station_id,0
5,istdaten,not convertible to timestamp,ANKUNFTSZEIT -> planned_arrival,0
6,istdaten,not convertible to timestamp,AN_PROGNOSE -> actual_arrival,0
7,istdaten,not convertible to timestamp,ABFAHRTSZEIT -> planned_departure,0
8,istdaten,not convertible to timestamp,AB_PROGNOSE -> actual_departure,0
9,istdaten,not convertible to boolean,DURCHFAHRT_TF -> is_pass_through,0


## 2. Problems found (profiling)

In [7]:
profile = OUTPUT_DIR / "profile"
for name in ["02_products", "04_arrival_status_trains", "05_flags_trains",
             "06_duplicates_trains", "07_delay_distribution_trains", "07b_delay_buckets_trains", "09_stations"]:
    print(name)
    display(pd.read_csv(profile / f"{name}.csv"))

02_products


,product,rows
0,Bus,29453068
1,Tram,3518030
2,Zug,2650747
3,BUS,684373
4,Metro,68088
5,Zahnradbahn,26645
6,Schiff,11652
7,NaN,175


04_arrival_status_trains


,arrival_status,rows
0,REAL,2068696
1,NaN,224829
2,PROGNOSE,183359
3,UNBEKANNT,173863


05_flags_trains


...

,stat,minutes
0,rows with planned+actual arrival,2.245635e+06
1,min,-1.444430e+03
2,p1,-1.766667e+00
3,p5,-9.000000e-01
4,median,5.333333e-01
5,mean,8.500000e-01
6,p95,3.700000e+00
7,p99,1.000000e+01
8,p99.9,3.900000e+01
9,max,1.456000e+03


07b_delay_buckets_trains


,bucket,rows
0,early > 10 min,434
1,early 0-10 min,543993
2,on time (< 3 min late),1535971
3,late 3-60 min,164285
4,late 1-3 h,901
5,late > 3 h,51


09_stations


,check,value
0,station list: rows,157737
1,station list: distinct station_id,73089
2,station list: rows still valid today,60902
3,station list: distinct station_id still valid ...,60584
4,train data: distinct stations,2246
5,train stations with NO match in the station list,600
6,train rows with NO match in the station list,246788


### Example problem: the same station appears several times in the station list

In [6]:
show(table("stations_typed").filter("station_id = 8507000")
          .select("station_id", "station_name", "valid_from", "valid_to", "canton"))

,station_id,station_name,valid_from,valid_to,canton
0,8507000,Bern,1987-12-31,2002-04-19,BE
1,8507000,Bern,2002-04-20,2018-02-17,BE
2,8507000,Bern,2018-02-18,2019-12-05,BE
3,8507000,Bern,2019-12-06,2019-12-19,BE
4,8507000,Bern,2019-12-20,2021-03-31,BE
5,8507000,Bern,2021-04-01,9999-12-31,BE


### Example problem: train arrivals with an unknown actual time

In [7]:
show(stops.filter("product = 'Zug' AND arrival_status = 'UNBEKANNT'")
          .select("operating_day", "line_text", "station_name", "planned_arrival", "actual_arrival", "arrival_status"), 8)

,operating_day,line_text,station_name,planned_arrival,actual_arrival,arrival_status
0,2026-10-02,R37,Les Breuleux-Eglise,2026-10-02 06:16:00,NaT,UNBEKANNT
1,2026-10-02,R37,Le Noirmont,2026-10-02 06:24:00,NaT,UNBEKANNT
2,2026-10-02,R37,Les Breuleux-Eglise,2026-10-02 06:44:00,NaT,UNBEKANNT
3,2026-10-02,R37,Le Noirmont,2026-10-02 06:52:00,NaT,UNBEKANNT
4,2026-10-02,R37,Les Breuleux-Eglise,2026-10-02 06:04:00,NaT,UNBEKANNT
5,2026-10-02,R37,Les Breuleux,2026-10-02 06:06:00,NaT,UNBEKANNT
6,2026-10-02,R37,Les Breuleux-Eglise,2026-10-02 07:16:00,NaT,UNBEKANNT
7,2026-10-02,R37,Le Noirmont,2026-10-02 07:24:00,NaT,UNBEKANNT


## 3. After cleaning
(run `src/02_clean.py` first)

In [9]:
pd.read_csv(OUTPUT_DIR / "cleaning_report.csv")

,table,step,rows_before,rows_after,removed,modified,reason
0,stations,0. after ingestion,157737,157737,0,0,NaN
1,stations,1. one row per station_id (latest version),157737,73089,84648,0,historical versions would duplicate train rows...
2,stations,2. impossible coordinates -> NULL,73089,73089,0,31,Swiss station with coordinates outside Switzer...
3,stops,0. after ingestion,36412778,36412778,0,0,"all transport types, 15 days"
4,stops,1. keep trains only,36412778,2650747,33762031,0,question is about the rail network; buses/tram...
5,stops,2. remove duplicate rows,2650747,2650680,67,0,fully identical rows
6,stops,3. remove pass-through (no stop),2650680,2649222,1458,0,train did not stop
7,stops,4. remove timestamps outside the day,2649222,2648434,788,0,planned time not on the operating day or the d...
8,stops,5. normalise 9-digit station IDs to 7 digits,2648434,2648434,0,0,9-digit IDs = station number + 2-digit platfor...
9,stops,6. keep Swiss stations only,2648434,2402588,245846,0,foreign stations of international trains (Hung...


In [10]:
clean = table("stops_clean")
print(f"clean rows: {clean.count():,}")
show(clean.filter("station_id = 8507000 AND operating_day = '2026-10-01'")
          .select("station_name", "line_text", "planned_arrival", "arrival_status",
                  "is_cancelled", "arrival_delay_min", "is_punctual", "canton")
          .orderBy("planned_arrival"))

clean rows: 2,402,588


,station_name,line_text,planned_arrival,arrival_status,is_cancelled,arrival_delay_min,is_punctual,canton
0,Bern,IR66,NaT,None,False,NaN,None,BE
1,Bern,IR65,NaT,None,False,NaN,None,BE
2,Bern,S7,NaT,None,False,NaN,None,BE
3,Bern,RE2,NaT,None,False,NaN,None,BE
4,Bern,S5,NaT,None,False,NaN,None,BE
5,Bern,IR35,NaT,None,False,NaN,None,BE
6,Bern,S7,NaT,None,False,NaN,None,BE
7,Bern,IR35,NaT,None,False,NaN,None,BE
8,Bern,S5,NaT,None,False,NaN,None,BE
9,Bern,S5,NaT,None,False,NaN,None,BE


### Same station after cleaning: only one version left

In [11]:
show(table("stations_clean").filter("station_id = 8507000")
          .select("station_id", "station_name", "valid_from", "valid_to", "canton"))

,station_id,station_name,valid_from,valid_to,canton
0,8507000,Bern,2021-04-01,9999-12-31,BE


## 4. First look at importance (trains per day, lines) and delay

In [12]:
from pyspark.sql import functions as F
show(clean.filter(~F.col("is_cancelled"))
          .groupBy("station_id", "station_name")
          .agg((F.countDistinct("operating_day", "trip_id") / F.countDistinct("operating_day")).alias("trains_per_day"),
               F.countDistinct("line_text").alias("lines"),
               F.round(F.avg("arrival_delay_min"), 2).alias("avg_delay_min"))
          .orderBy(F.desc("trains_per_day")), 15)

,station_id,station_name,trains_per_day,lines,avg_delay_min
0,8503000,Zürich HB,1927.133333,61,1.05
1,8507000,Bern,1485.066667,39,0.47
2,8506000,Winterthur,979.866667,40,0.52
3,8503006,Zürich Oerlikon,901.600000,27,1.07
4,8500218,Olten,812.733333,32,0.64
5,8505000,Luzern,807.066667,31,0.40
6,8500010,Basel SBB,796.066667,29,1.28
7,8502113,Aarau,708.866667,21,0.72
8,8503020,Zürich Hardbrücke,707.666667,18,1.13
9,8503003,Zürich Stadelhofen,699.800000,19,0.78
